# 13 · Which features matter? L1, L2 and elastic-net regularization (with robustness checks)

**Goal:** give regularized logistic regression every legal feature, find which carry weight for each model, and save
ranked feature sets (`reports/phase2/experiments/feature_sets_master.json`) that the model notebooks train on.
Because those choices drive every later model, each result is checked several ways before it is trusted.

| | Pre-call model | Post-call model |
|---|---|---|
| Rows | synthetic **train**, `label_usable` | + `post_call_row` |
| Target | `paid_30d` | `paid_after_call_30d` |
| Candidates | 21 history fields | the 21 + 21 call fields = 42 |
| Never used | answer-like columns, ids, dates, bookkeeping, prior-contact fields (not in Salud's data), `ever_paid` (exact copy of the automatic "never paid" flag) | same |

Synthetic **test** and **real** rows are not used to choose anything. Real rows appear only in section 8, to check each
feature's direction (with confidence intervals).

| Section | What it answers | Guards against |
|---|---|---|
| 0 Guards | right rows, no answer-like columns, no exact duplicates | leakage, wrong inputs |
| 1–2 | preprocessing; near-duplicate pairs | scale effects; misreading L1 |
| 3 | penalty strength by grouped CV (wide grid) | choosing at a grid edge |
| 4 | coefficient paths | — |
| 5 | L1 / L2 / elastic-net weights | — |
| 6 | stability selection (100 half-samples of patients) | L1 picking one twin at random |
| 7 | permutation importance, **logistic regression and LightGBM** | importance only a linear model sees (or misses) |
| 8 | direction in real data, with 95% CIs | synthetic-only effects |
| 9 | combined ranking → feature sets | relying on one method |
| 10 | accuracy per set, LR and LightGBM | sets that only suit one model family |
| 11 | **nested** selection (choose inside each fold) | optimistic accuracy for chosen sets |
| 12 | the ranking with other random seeds | seed-dependent answers |

**Caveat:** synthetic sentiment and QA were sampled from the call outcome only (notebook 12), so they cannot show value
beyond the outcome here. Importance means "useful for prediction in the synthetic world tuned to Salud", not "causes payment".

In [ ]:
%matplotlib inline
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.inspection import permutation_importance
from salud_pipeline.phase2.propensity import dictionary as D, modeling as M
from salud_pipeline.phase2.propensity.master import MASTER_XLSX
from salud_pipeline.paths import PHASE2_REPORTS

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)
C1, C2, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"     # validated pair
POS, NEG = C1, C2
PAL8 = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]  # fixed order
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titlecolor": INK, "legend.frameon": False, "font.size": 9})
SEED = 42

m = pd.read_excel(MASTER_XLSX)
PRE_CAND = [f for f in D.PRE_FEATURES if f not in M.NO_TRANSFER + ["ever_paid"]] + D.NEW_PRE_CANDIDATES
POST_CAND = PRE_CAND + D.MODEL_POST_EXTRA
for c in set(PRE_CAND + POST_CAND) | {"ever_paid", "paid_30d", "paid_after_call_30d", "paid_on_call"}:
    m[c] = pd.to_numeric(m[c].astype(float) if m[c].dtype == bool else m[c], errors="coerce")
u = m[m.label_usable]
train = u[(u.data_type == "synthetic") & (u.split == "train")]
real = u[u.data_type == "real"]
DATA = {"pre-call": dict(df=train, feats=PRE_CAND, target="paid_30d", real=real),
        "post-call": dict(df=train[train.post_call_row], feats=POST_CAND, target="paid_after_call_30d",
                          real=real[real.post_call_row])}
for k, d in DATA.items():
    print(f"{k}: {len(d['df']):,} training rows, {d['df'].guarantor_id.nunique():,} patients, "
          f"{len(d['feats'])} candidates, pay rate {d['df'][d['target']].mean():.2f}; real check rows {len(d['real'])}")

**Result:** pre-call: 6,647 training calls from 1,477 patients, 21 candidates, pay rate 0.57. Post-call: 5,368 calls
from 1,421 patients, 42 candidates, pay rate 0.46. Real rows for the direction check: 58 pre-call, 47 post-call.

## 0 · Guards (the notebook stops if any fails)

In [ ]:
ANSWER_LIKE = {"paid_30d", "paid_after_call_30d", "paid_amount_30d", "days_to_first_payment", "paid_fraction_30d",
               "paid_on_call", "payment_on_call", "label_usable", "label_reliable", "window_complete"}
BOOKKEEPING = {"data_type", "split", "post_call_row", "sentiment_source", "qa_source", "guarantor_id", "call_id",
               "snapshot_id", "as_of", "agent", "tcn_record_id", "source"}
post_df = DATA["post-call"]["df"]
checks = {
    "no answer-like or bookkeeping column among candidates": not (set(POST_CAND) & (ANSWER_LIKE | BOOKKEEPING)),
    "no prior-contact field (absent from Salud's data)": not (set(POST_CAND) & set(M.NO_TRANSFER)),
    "pre-call candidates contain no call field": not (set(PRE_CAND) & set(D.MODEL_POST_EXTRA)),
    "training rows: synthetic train only": set(train.data_type) == {"synthetic"} and set(train.split) == {"train"},
    "training rows: label usable": bool(train.label_usable.all()),
    "post-call rows: no payment on the call": not post_df.paid_on_call.astype(bool).any(),
    "ever_paid is an exact copy of the never-paid flag (so it is dropped)":
        bool((train.ever_paid == train.days_since_last_payment.notna().astype(float)).all()),
    "no exact duplicate columns left among candidates": not post_df[POST_CAND].T.duplicated().any(),
    "no candidate is constant in training": bool((post_df[POST_CAND].nunique() > 1).all() and (train[PRE_CAND].nunique() > 1).all()),
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)
assert all(checks.values()), "a guard failed"

**Result:** all 9 guards pass. No answer-like, bookkeeping or prior-contact column is a candidate; only synthetic
training rows with a usable label are used; `ever_paid` is an exact copy of the never-paid flag, so it is dropped; no
exact duplicate or constant column is left.

## 1 · Preprocessing (fitted inside every fold)

`days_since_last_payment` is empty when the patient never paid: filled with "longer ago than anyone" plus a `__never`
flag (`M.RecencyFill`). log1p for non-negative skewed fields, then standardize, so weights mean "effect of 1 SD".

In [ ]:
BOUNDED = {"paid_share_so_far"} | set(D.MODEL_POST_EXTRA) - {"talk_seconds", "n_actions"}  # flags, sentiment, QA, shares
def log_cols(feats): return [f for f in feats if f not in BOUNDED]
def prep(feats):
    return [("fill", M.RecencyFill()), ("log1p", M._log1p_cols(log_cols(feats))), ("scale", StandardScaler())]
def lr(penalty, C, l1_ratio=None):
    if penalty == "l1":
        return LogisticRegression(penalty="l1", C=C, solver="liblinear", max_iter=5000, random_state=SEED)
    if penalty == "en":
        return LogisticRegression(penalty="elasticnet", C=C, l1_ratio=l1_ratio, solver="saga", max_iter=5000,
                                  random_state=SEED)
    return LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=5000)
def pipe(feats, penalty, C, l1_ratio=None):
    return M.pipeline(prep(feats), ("model", lr(penalty, C, l1_ratio)))
def gbm(feats):   # fixed, moderate settings: a nonlinear cross-check, not a tuned model
    return M.pipeline([("fill", M.RecencyFill())], ("model", M.lgbm(None, n_estimators=300, learning_rate=0.03,
                                                                     num_leaves=15, min_child_samples=30)))
d = DATA["pre-call"]
Xt = M.pipeline(prep(d["feats"])).fit_transform(d["df"][d["feats"]])
pd.DataFrame({"skew before": d["df"][d["feats"]].skew(), "skew after": Xt[d["feats"]].skew()}).round(2).T

**Result:** log1p brings the skew of money and count fields from up to 17 (`last_payment_amount`) and 16
(`amount_adjustments_365d`) down to 2.3 or less. What remains is in zero-heavy fields (insurance 2.2, adjustments 2.0,
`paid_share_so_far` 2.3, which is a 0–1 share and not logged).

## 2 · Near-duplicates among the candidates (|Spearman ρ| > 0.8)

When two features carry the same information, L1 keeps one (often at random), L2 splits the weight, and permutation
importance undercounts both (shuffling one, the twin still carries the signal). Read every later section with this table.

In [ ]:
def dup_pairs(df, feats, thr=0.8):
    c = df[feats].corr("spearman").abs()
    out = [(a, b, c.loc[a, b]) for i, a in enumerate(feats) for b in feats[i + 1:] if c.loc[a, b] > thr]
    return pd.DataFrame(out, columns=["feature A", "feature B", "|rho|"]).sort_values("|rho|", ascending=False)
pairs = dup_pairs(post_df, POST_CAND)
TWINS = {f: sorted(set(pairs.loc[pairs["feature A"] == f, "feature B"]) | set(pairs.loc[pairs["feature B"] == f, "feature A"]))
         for f in POST_CAND}
print(f"{len(pairs)} near-duplicate pairs"); pairs.round(2).reset_index(drop=True)

**Result:** 31 pairs with |ρ| > 0.8, **all inside the pre-call history fields** (no call field has a near-duplicate):
- one **payment-history family**: payment counts (90d, 365d, ever), amounts paid (90d, 365d, ever), `paid_share_so_far`,
  `last_payment_amount` — ρ 0.84–1.00 with each other (365d count = ever count: 1.00);
- insurance count ≈ amount (0.99); adjustment count ≈ amount (0.99); placement balance ≈ balance on the call day (0.96).

## 3 · Penalty strength: grouped 5-fold CV over a wide grid

`C` = inverse penalty (small = strong). The grid spans 10⁻⁴…10², so a chosen value at the edge would be visible.
L1 uses the **one-SE rule** (fewest features within one SE of the best). L2 uses its **best** C: at very strong L2
penalties every weight collapses towards "the feature on its own", which is no longer a multivariate answer.

In [ ]:
CS = np.logspace(-4, 2, 19)
def folds(df, target, k=5, seed=SEED):
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
                .split(df, df[target].astype(int), df["guarantor_id"]))
def cv_auc(d, make, feats=None, fold_list=None):
    feats = feats or d["feats"]; X, y = d["df"][feats], d["df"][d["target"]].astype(int).to_numpy()
    out = []
    for tr, te in (fold_list or d["folds"]):
        p = make().fit(X.iloc[tr], y[tr]); out.append(roc_auc_score(y[te], p.predict_proba(X.iloc[te])[:, 1]))
    return np.array(out)
def cv_curve(d, penalty, fold_list=None):
    X, y = d["df"][d["feats"]], d["df"][d["target"]].astype(int).to_numpy(); rows = []
    for C in CS:
        aucs, nz = [], []
        for tr, te in (fold_list or d["folds"]):
            p = pipe(d["feats"], penalty, C).fit(X.iloc[tr], y[tr])
            aucs.append(roc_auc_score(y[te], p.predict_proba(X.iloc[te])[:, 1])); nz.append((np.abs(p[-1].coef_[0]) > 1e-8).sum())
        rows.append({"C": C, "auc": np.mean(aucs), "se": np.std(aucs, ddof=1) / np.sqrt(len(aucs)), "n_nonzero": np.mean(nz)})
    return pd.DataFrame(rows)
def one_se(curve):
    best = curve.loc[curve.auc.idxmax()]
    return curve[curve.auc >= best.auc - best.se].iloc[0]

for name, d in DATA.items():
    d["folds"] = folds(d["df"], d["target"])
    d["curve_l1"], d["curve_l2"] = cv_curve(d, "l1"), cv_curve(d, "l2")
    d["C_l1"] = one_se(d["curve_l1"]).C
    d["C_l2"] = d["curve_l2"].loc[d["curve_l2"].auc.idxmax()].C

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (name, d) in zip(axes, DATA.items()):
    for curve, col, lab in [(d["curve_l1"], C1, "L1 (lasso)"), (d["curve_l2"], C2, "L2 (ridge)")]:
        ax.plot(curve.C, curve.auc, color=col, lw=2, marker="o", ms=4, label=lab)
        ax.fill_between(curve.C, curve.auc - curve.se, curve.auc + curve.se, color=col, alpha=0.15, lw=0)
    ax.axvline(d["C_l1"], color=C1, ls=":", lw=1); ax.axvline(d["C_l2"], color=C2, ls=":", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("C (left = stronger penalty)"); ax.set_ylabel("CV AUC")
    ax.set_title(f"{name}: AUC vs penalty (dotted = chosen)"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()
for name, d in DATA.items():
    b1 = d["curve_l1"].set_index("C").loc[d["C_l1"]]; b2 = d["curve_l2"].set_index("C").loc[d["C_l2"]]
    edge = lambda C: " (AT GRID EDGE)" if C in (CS[0], CS[-1]) else ""
    print(f"{name}: L1 one-SE C={d['C_l1']:.4g}{edge(d['C_l1'])} -> AUC {b1.auc:.3f}, ~{b1.n_nonzero:.0f} features | "
          f"L2 best C={d['C_l2']:.4g}{edge(d['C_l2'])} -> AUC {b2.auc:.3f}")

**Result:** pre-call: L1 one-SE C = 0.0215 → AUC 0.702 with ~7 features; L2 best C = 46 → 0.708. Post-call: L1
one-SE C = 0.0215 → 0.796 with ~12 features; L2 best C = 0.22 → 0.797. No choice is at the edge of the grid. AUC is flat
over a wide range of C: the penalty changes which features are used much more than how well the model ranks.

## 4 · Coefficient paths

Each line = one feature's weight as the penalty weakens. Under L1 features enter one by one: **the earlier a feature
leaves zero, the more it matters.**

In [ ]:
PATH_CS = np.logspace(-4, 1.5, 45)
def paths(d, penalty):
    X, y = d["df"][d["feats"]], d["df"][d["target"]].astype(int); rows = []
    for C in PATH_CS:
        p = pipe(d["feats"], penalty, C).fit(X, y)
        rows.append(pd.Series(p[-1].coef_[0], index=p[-1].feature_names_in_, name=C))
    return pd.DataFrame(rows)
def entry_order(path):
    nz = path.abs() > 1e-6
    return nz.idxmax().where(nz.any(), np.inf).sort_values()

for name, d in DATA.items():
    d["path_l1"], d["path_l2"] = paths(d, "l1"), paths(d, "l2")
    d["entry"] = entry_order(d["path_l1"])
    top = list(d["entry"].index[:8])
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    for ax, (path, lab) in zip(axes, [(d["path_l1"], "L1"), (d["path_l2"], "L2")]):
        for f in sorted(path.columns, key=lambda c: c in top):
            ax.plot(path.index, path[f], lw=2 if f in top else 1, color=PAL8[top.index(f)] if f in top else GRID,
                    label=f if f in top else None)
        ax.set_xscale("log"); ax.axhline(0, color=MUTED, lw=0.8); ax.set_xlabel("C (left = stronger penalty)")
        ax.set_title(f"{name}: {lab} path (first 8 to enter L1 in colour)")
    axes[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5)); plt.tight_layout(); plt.show()
    print(f"{name}: L1 entry order: " + ", ".join(f"{i+1}. {f}" for i, f in enumerate(d["entry"].index[:15])))

**Result (L1 entry order):**
- Pre-call: 1 `n_payments_90d`, 2 `days_since_last_payment`, 3 `paid_share_so_far`, 4 `days_since_first_placement`,
  5 `n_payments_365d`, 6 `n_physician_accounts`, 7 `amount_adjustments_365d`, 8 `days_since_last_placement`, …
- Post-call: 1 `n_payments_90d`, 2 `action_payment_plan`, 3 `days_since_last_payment`, 4 `promised_to_pay`,
  5 `spoke_with_agent`, 6 `days_since_first_placement`, 7 `action_shared_balance`, 8 `paid_share_so_far`, …

## 5 · Weights at the chosen penalty: L1, L2, elastic net

+ = higher value → more likely to pay. Elastic net: `l1_ratio` and C tuned by grouped CV. **Opposite-sign weights on
near-duplicate pairs (section 2) cancel out and should not be read individually.**

In [ ]:
def coefs(d):
    X, y = d["df"][d["feats"]], d["df"][d["target"]].astype(int)
    grid = [(C, r) for r in (0.2, 0.5, 0.8) for C in CS[4:15:2]]
    scores = {(C, r): cv_auc(d, lambda C=C, r=r: pipe(d["feats"], "en", C, r)).mean() for C, r in grid}
    d["C_en"], d["r_en"] = max(scores, key=scores.get)
    out = {}
    for lab, (pen, C, r) in {"L1": ("l1", d["C_l1"], None), "L2": ("l2", d["C_l2"], None),
                             "elastic net": ("en", d["C_en"], d["r_en"])}.items():
        p = pipe(d["feats"], pen, C, r).fit(X, y); out[lab] = pd.Series(p[-1].coef_[0], index=p[-1].feature_names_in_)
    t = pd.DataFrame(out)
    return t.reindex(t["elastic net"].abs().sort_values(ascending=False).index)

for name, d in DATA.items():
    d["coefs"] = t = coefs(d)
    fig, ax = plt.subplots(figsize=(7, 0.22 * len(t) + 1)); yy = np.arange(len(t))
    ax.barh(yy, t["elastic net"], color=[POS if v > 0 else NEG for v in t["elastic net"]], height=0.7)
    ax.plot(t["L1"], yy, "o", ms=5, color=INK, label="L1 (0 = dropped)")
    ax.set_yticks(yy, t.index, fontsize=8); ax.invert_yaxis(); ax.axvline(0, color=MUTED, lw=0.8)
    ax.set_xlabel("standardized weight (bars: elastic net; blue = pays more, orange = pays less)")
    ax.set_title(f"{name}: weight of each feature"); ax.legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()
    print(f"{name}: elastic net C={d['C_en']:.4g}, l1_ratio={d['r_en']}; non-zero: L1 {int((t['L1'].abs() > 1e-8).sum())}, "
          f"elastic net {int((t['elastic net'].abs() > 1e-8).sum())} of {len(t)} columns")
    display(t.round(3))

**Result:** L1 keeps 7 pre-call and 14 post-call columns. The tuned elastic net (pre-call l1_ratio 0.2, post-call 0.8)
keeps all 22 pre-call columns (21 + the never-paid flag) and 35 of 43 post-call ones, so it does not narrow the pre-call set.
Directions that are consistent across L1, L2 and elastic net: more recent payments (+), longer since the last payment (−),
payment plan set up on the call (+), promise to pay (+), older debt (−). Near-duplicates take **opposite** signs
(pre-call L2: `amount_paid_365d` +0.52, `amount_paid_90d` −0.83, `n_payments_ever` −0.24): they cancel and must not be
read one by one.

## 6 · Stability selection

L1 at the one-SE penalty on 100 half-samples **of patients**. Kept ≥ 80% = stable. Shown next to each feature: its
near-duplicates, because a ~50% feature is often sharing the role with a twin.

In [ ]:
def stability(d, C, n=100, frac=0.5, seed=SEED):
    rng = np.random.default_rng(seed); pats = d["df"].guarantor_id.unique(); counts = 0
    for _ in range(n):
        keep = set(rng.choice(pats, size=int(frac * len(pats)), replace=False))
        s = d["df"][d["df"].guarantor_id.isin(keep)]
        p = pipe(d["feats"], "l1", C).fit(s[d["feats"]], s[d["target"]].astype(int))
        counts = counts + pd.Series(np.abs(p[-1].coef_[0]) > 1e-8, index=p[-1].feature_names_in_).astype(int)
    return (counts / n).sort_values(ascending=False)

for name, d in DATA.items():
    d["stab"] = s = stability(d, d["C_l1"])
    fig, ax = plt.subplots(figsize=(7, 0.22 * len(s) + 1)); yy = np.arange(len(s))
    ax.barh(yy, s, color=[C1 if v >= 0.8 else GRID for v in s], height=0.7)
    ax.axvline(0.8, color=MUTED, ls=":", lw=1); ax.set_xlim(0, 1); ax.set_yticks(yy, s.index, fontsize=8); ax.invert_yaxis()
    ax.set_xlabel("share of 100 half-samples where L1 keeps it (blue = stable)"); ax.set_title(f"{name}: stability")
    plt.tight_layout(); plt.show()
    tw = pd.DataFrame({"kept share": s, "near-duplicates": [", ".join(TWINS.get(f, [])) for f in s.index]})
    display(tw[tw["kept share"] > 0].round(2))

**Result:**
- Pre-call, stable (kept in 100% of 100 half-samples): `n_payments_90d`, `days_since_last_payment`. Next: `n_payments_365d`
  0.42, `paid_share_so_far` 0.35, `days_since_first_placement` 0.30, `n_physician_accounts` 0.26 — the first two share the
  role with their payment-history twins. Everything else ≤ 0.18.
- Post-call, stable: `promised_to_pay`, `action_payment_plan`, `n_payments_90d`, `days_since_last_payment` (1.00 each),
  `days_since_first_placement` (0.95). Close: `spoke_with_agent` 0.78, `action_shared_balance` 0.69. Sentiment and QA ≤ 0.10.

## 7 · Permutation importance: logistic regression **and** LightGBM

AUC lost on held-out folds when one feature is shuffled. LightGBM is a nonlinear cross-check: a feature important to
the trees but not to logistic regression has a curved or interaction effect.

In [ ]:
def perm(d, make, repeats=5):
    X, y = d["df"][d["feats"]], d["df"][d["target"]].astype(int); res = []
    for tr, te in d["folds"]:
        p = make().fit(X.iloc[tr], y.iloc[tr])
        r = permutation_importance(p, X.iloc[te], y.iloc[te], scoring="roc_auc", n_repeats=repeats, random_state=SEED)
        res.append(pd.Series(r.importances_mean, index=d["feats"]))
    return pd.concat(res, axis=1).mean(axis=1)

for name, d in DATA.items():
    d["perm_lr"] = perm(d, lambda: pipe(d["feats"], "l2", d["C_l2"]))
    d["perm_gbm"] = perm(d, lambda: gbm(d["feats"]))
    d["auc_lr_all"] = cv_auc(d, lambda: pipe(d["feats"], "l2", d["C_l2"])).mean()
    d["auc_gbm_all"] = cv_auc(d, lambda: gbm(d["feats"])).mean()
    pi = pd.DataFrame({"LR": d["perm_lr"], "LightGBM": d["perm_gbm"]}).sort_values("LightGBM", ascending=False)
    top = pi.head(15)
    fig, ax = plt.subplots(figsize=(7, 4)); yy = np.arange(len(top))
    ax.barh(yy - 0.2, top["LR"], height=0.38, color=C1, label="logistic regression")
    ax.barh(yy + 0.2, top["LightGBM"], height=0.38, color=C2, label="LightGBM")
    ax.set_yticks(yy, top.index, fontsize=8); ax.invert_yaxis(); ax.set_xlabel("AUC lost when shuffled")
    ax.set_title(f"{name}: permutation importance (top 15)"); ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.show()
    print(f"{name}: CV AUC with all candidates: LR {d['auc_lr_all']:.3f}, LightGBM {d['auc_gbm_all']:.3f}")
    nl = pi[(pi["LightGBM"] > 0.005) & (pi["LightGBM"] > 2 * pi["LR"].clip(lower=0))]
    print("  matter more to LightGBM than to LR (curved / interaction effects):", ", ".join(nl.index) or "none")

**Result:** with all candidates, logistic regression and LightGBM are level (pre-call 0.708 vs 0.705; post-call 0.797
vs 0.795). The two models lean on different members of the payment-history family:
- logistic regression: `n_payments_90d` (AUC lost 0.27 pre-call, 0.20 post-call), then `amount_paid_365d` / `action_payment_plan`;
- LightGBM: `days_since_last_payment` (0.10 pre-call, 0.13 post-call), then `n_payments_90d`, `days_since_last_placement`,
  `last_payment_amount` (pre-call) and `action_payment_plan` (post-call).
`days_since_last_payment` and `last_payment_amount` matter more to the trees than to the linear model (curved effects).

## 8 · Direction in real data (with 95% bootstrap intervals)

Each feature alone, as AUC for the target (> 0.5 = higher value → pays more). Real: 58 pre-call / 47 post-call rows, so
intervals are wide; only an interval that excludes 0.5 on the **opposite side** to synthetic is a real contradiction.

In [ ]:
def alone(df, f, target):
    x, y = df[f], df[target].astype(int)
    x = x.fillna(x.max() + 1 if f == "days_since_last_payment" else 0)
    return roc_auc_score(y, x) if x.nunique() > 1 and y.nunique() > 1 else np.nan
def boot_ci(df, f, target, n=1000, seed=SEED):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        v = alone(df.iloc[rng.integers(0, len(df), len(df))], f, target)
        if not np.isnan(v): vals.append(v)
    return np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)

for name, d in DATA.items():
    rows = []
    for f in d["feats"]:
        a_s, a_r = alone(d["df"], f, d["target"]), alone(d["real"], f, d["target"])
        lo, hi = boot_ci(d["real"], f, d["target"])
        side_s = np.sign(a_s - 0.5) if abs(a_s - 0.5) > 0.02 else 0
        side_r = 1 if lo > 0.5 else (-1 if hi < 0.5 else 0)
        rows.append({"feature": f, "synthetic": a_s, "real": a_r, "real 95% low": lo, "real 95% high": hi,
                     "real clearly agrees": side_r != 0 and side_r == side_s,
                     "real clearly contradicts": side_r != 0 and side_s != 0 and side_r != side_s})
    d["real_check"] = rc = pd.DataFrame(rows).set_index("feature")
    print(f"{name}: clearly confirmed by real data: {', '.join(rc.index[rc['real clearly agrees']]) or 'none'}")
    print(f"{name}: clearly contradicted by real data: {', '.join(rc.index[rc['real clearly contradicts']]) or 'none'}")
display(DATA["pre-call"]["real_check"].round(3))

**Result:** confirmed in Salud's real calls (95% interval excludes 0.5, same side as synthetic): the whole
payment-history family — `n_payments_90d` 0.72 (0.62–0.83), `n_payments_365d` 0.73, the amounts paid, `paid_share_so_far`
0.72, `last_payment_amount` 0.73 and `days_since_last_payment` 0.28 (0.17–0.39, i.e. longer ago → pays less).
**Contradicted: none.** Not confirmed (interval includes 0.5): account counts, balances, debt age (`days_since_first_placement`
0.64, 0.48–0.78), insurance, adjustments, days since discharge, and every call field (only 47 post-call rows).

## 9 · Combined ranking and feature sets

Rank by four views that are robust to near-duplicates and to model form: **L1 entry order, stability, permutation (LR),
permutation (LightGBM)**; combined = their mean. (Raw L2 / elastic-net weight size is not used: twins split it.)

| Set | Meaning |
|---|---|
| `all` | every candidate |
| `current` | the fields the current models use (11 pre-call, 32 post-call) |
| `l1_one_se` | non-zero in L1 at the one-SE penalty |
| `stable` | kept by L1 in ≥ 80% of half-samples |
| `elastic_net` | non-zero in the tuned elastic net |
| `top5`, `top10`, `top15` | best by combined rank |

In [ ]:
def ranking(d, entry, stab, p_lr, p_gbm):
    base = d["feats"]
    r = pd.DataFrame({"rank L1 entry": entry.reindex(base).replace(np.inf, np.nan).rank(na_option="bottom"),
                      "rank stability": stab.reindex(base).rank(ascending=False),
                      "rank perm LR": p_lr.reindex(base).rank(ascending=False),
                      "rank perm LightGBM": p_gbm.reindex(base).rank(ascending=False)})
    r["combined"] = r.mean(axis=1)
    return r.sort_values(["combined", "rank perm LightGBM"])

SETS = {}
for name, d in DATA.items():
    d["rank"] = r = ranking(d, d["entry"], d["stab"], d["perm_lr"], d["perm_gbm"])
    r["stability"] = d["stab"].reindex(r.index); r["perm LR"] = d["perm_lr"].reindex(r.index)
    r["perm LightGBM"] = d["perm_gbm"].reindex(r.index); r["elastic-net weight"] = d["coefs"]["elastic net"].reindex(r.index)
    r["real clearly agrees"] = d["real_check"]["real clearly agrees"].reindex(r.index)
    order = list(r.index); nz = lambda col: [f for f in order if abs(d["coefs"][col].get(f, 0)) > 1e-8]
    SETS[name] = {"all": d["feats"],
                  "current": D.MODEL_PRE if name == "pre-call" else D.MODEL_PRE + D.MODEL_POST_EXTRA,
                  "l1_one_se": nz("L1"), "stable": [f for f in order if d["stab"].get(f, 0) >= 0.8],
                  "elastic_net": nz("elastic net"), "top5": order[:5], "top10": order[:10], "top15": order[:15]}
    print(f"{name}: combined ranking"); display(r.round(3))
pd.DataFrame({name: {k: len(v) for k, v in s.items()} for name, s in SETS.items()})

**Result (combined rank):**
- Pre-call top 10: `n_payments_90d`, `days_since_last_payment`, `n_payments_365d`, `amount_adjustments_365d`,
  `days_since_first_placement`, `days_since_last_placement`, `paid_share_so_far`, `n_physician_accounts`, `amount_paid_365d`, `n_accounts`.
- Post-call top 10: `n_payments_90d`, `days_since_last_payment`, `action_payment_plan`, `promised_to_pay`,
  `days_since_last_placement`, `days_since_first_placement`, `paid_share_so_far`, `amount_adjustments_365d`, `spoke_with_agent`, `n_payments_365d`.
- Set sizes (pre / post): all 21 / 42, current 11 / 32, l1_one_se 7 / 14, stable 2 / 5, elastic_net 21 / 34, top5/10/15.
Note: `amount_adjustments_365d` reaches the pre-call top 5 on permutation importance, but it is stable in only 18% of
half-samples and not confirmed in real data — treat it as tier 3 (see the findings).

## 10 · Accuracy per set: logistic regression and LightGBM (grouped 5-fold CV, training rows)

LR gets its own best C per set. **These numbers are optimistic for the chosen sets** (chosen and scored on the same rows);
section 11 gives the honest version.

In [ ]:
rows = []
for name, d in DATA.items():
    for set_name, feats in SETS[name].items():
        best = max(((C, cv_auc(d, lambda C=C: pipe(feats, "l2", C), feats)) for C in CS[2:17:2]), key=lambda t: t[1].mean())
        g = cv_auc(d, lambda: gbm(feats), feats)
        rows.append({"model": name, "feature set": set_name, "n": len(feats), "LR AUC": best[1].mean(),
                     "LR SE": best[1].std(ddof=1) / np.sqrt(5), "LightGBM AUC": g.mean(), "LightGBM SE": g.std(ddof=1) / np.sqrt(5)})
sets_cv = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (name, g) in zip(axes, sets_cv.groupby("model", sort=False)):
    g = g.sort_values("n").reset_index(drop=True); yy = np.arange(len(g))
    ax.errorbar(g["LR AUC"], yy - 0.15, xerr=g["LR SE"], fmt="o", color=C1, ms=6, capsize=3, label="logistic regression")
    ax.errorbar(g["LightGBM AUC"], yy + 0.15, xerr=g["LightGBM SE"], fmt="s", color=C2, ms=6, capsize=3, label="LightGBM")
    ax.set_yticks(yy, [f"{s} ({n})" for s, n in zip(g["feature set"], g["n"])], fontsize=8); ax.invert_yaxis()
    ax.set_xlabel("CV AUC (± 1 SE)"); ax.set_title(f"{name}: accuracy per feature set (number of features)")
axes[0].legend(fontsize=8, loc="lower left", bbox_to_anchor=(0, -0.42), ncol=2)
plt.tight_layout(); plt.show()
sets_cv.round(3)

**Result (optimistic: chosen and scored on the same rows):** pre-call LR 0.692 (stable, 2) · 0.706 (top5) · 0.703
(l1_one_se, 7) · 0.700 (current, 11) · 0.710 (top15) · 0.708 (all, 21); LightGBM 0.692–0.706. Post-call LR 0.791–0.799,
LightGBM 0.790–0.802 (best: l1_one_se, 14 features). SE ≈ 0.01, so on these numbers every set is within about two SE of
every other.

## 11 · Honest accuracy: nested selection

For each outer fold, the **whole selection** (one-SE L1 by inner grouped CV, and the L1 entry order) is redone on the outer
training part only, then scored on the outer held-out part. Compare with section 10: a large drop would mean the chosen
sets only looked good because they were chosen on the same rows. Also shown: how often each feature lands in the nested
top 5 (selection consistency).

In [ ]:
def nested(d, k_top=5):
    df, y_all = d["df"], d["df"][d["target"]].astype(int).to_numpy()
    res, picks = [], []
    for tr, te in d["folds"]:
        inner = dict(df=df.iloc[tr], feats=d["feats"], target=d["target"])
        inner["folds"] = folds(inner["df"], d["target"], seed=SEED + 1)
        C1se = one_se(cv_curve(inner, "l1", inner["folds"])).C
        ytr = y_all[tr]
        l1 = pipe(d["feats"], "l1", C1se).fit(df.iloc[tr][d["feats"]], ytr)
        l1_set = [f for f, c in zip(l1[-1].feature_names_in_, l1[-1].coef_[0]) if abs(c) > 1e-8 and f in d["feats"]]
        top = [f for f in entry_order(paths(inner, "l1")).index if f in d["feats"]][:k_top]
        picks.append(top)
        for set_name, feats in [("all", d["feats"]), ("nested l1_one_se", l1_set), (f"nested top{k_top}", top)]:
            for mdl, make in [("LR", lambda f=feats: pipe(f, "l2", 1.0)), ("LightGBM", lambda f=feats: gbm(f))]:
                p = make().fit(df.iloc[tr][feats], ytr)
                res.append({"set": set_name, "model": mdl, "n": len(feats),
                            "auc": roc_auc_score(y_all[te], p.predict_proba(df.iloc[te][feats])[:, 1])})
    r = pd.DataFrame(res).groupby(["set", "model"]).agg(n=("n", "mean"), auc=("auc", "mean"),
                                                        se=("auc", lambda a: a.std(ddof=1) / np.sqrt(len(a))))
    freq = pd.Series([f for p in picks for f in p]).value_counts() / len(picks)
    return r, freq

for name, d in DATA.items():
    d["nested"], d["nested_freq"] = nested(d)
    print(f"{name}: nested (honest) AUC"); display(d["nested"].round(3))
    print(f"{name}: share of outer folds where a feature is in the nested top 5:")
    display(d["nested_freq"].round(2).to_frame("share"))

**Result (honest):** with the selection redone inside each fold,
| | Pre-call LR / LightGBM | Post-call LR / LightGBM |
|---|---|---|
| all candidates | 0.707 / 0.705 | 0.797 / 0.795 |
| nested L1 one-SE (≈ 13 / 9 features) | 0.705 / 0.698 | 0.790 / 0.793 |
| nested top 5 | 0.692 / 0.692 | 0.782 / 0.783 |

So the top 5 that looked as good as all features (0.706) **loses about 0.015 AUC** when chosen honestly. Nested top 5
picks: pre-call `n_payments_90d` and `days_since_last_payment` in 5/5 folds, `paid_share_so_far` 4/5,
`days_since_first_placement` and `n_payments_365d` 3/5; post-call the same five in every fold (`n_payments_90d`,
`days_since_last_payment`, `action_payment_plan`, `promised_to_pay`, `spoke_with_agent`).

## 12 · Does the ranking depend on the random seed?

The ranking is recomputed with three other seeds (different CV folds and half-samples). Overlap with the main ranking
(1.0 = the same features).

In [ ]:
def rank_with_seed(d, seed):
    dd = dict(d); dd["folds"] = folds(d["df"], d["target"], seed=seed)
    C = one_se(cv_curve(dd, "l1", dd["folds"])).C
    stab = stability(dd, C, n=50, seed=seed)
    p_lr = perm(dd, lambda: pipe(d["feats"], "l2", d["C_l2"]), repeats=3)
    p_gbm = perm(dd, lambda: gbm(d["feats"]), repeats=3)
    return ranking(dd, d["entry"], stab, p_lr, p_gbm)

rows = []
for name, d in DATA.items():
    main10, main5 = set(d["rank"].index[:10]), set(d["rank"].index[:5])
    for seed in (7, 123, 2024):
        r = rank_with_seed(d, seed)
        rows.append({"model": name, "seed": seed, "top-5 overlap": len(main5 & set(r.index[:5])) / 5,
                     "top-10 overlap": len(main10 & set(r.index[:10])) / 10, "top 5 with this seed": ", ".join(r.index[:5])})
seed_check = pd.DataFrame(rows)
for _, r in seed_check.iterrows():
    print(f"{r['model']} seed {r['seed']}: {r['top 5 with this seed']}")
seed_check.drop(columns='top 5 with this seed')

**Result:** with seeds 7, 123 and 2024 the top 5 keeps 4 or 5 of its 5 features (overlap 0.8–1.0) and the top 10
keeps 8–10 of 10. Pre-call, `n_payments_90d` and `days_since_last_payment` are 1st and 2nd under every seed, and
`n_payments_365d` and `amount_adjustments_365d` are always in the top 5; the fifth place changes (`n_physician_accounts`,
`n_adjustments_365d`, `paid_share_so_far`). Post-call, `n_payments_90d`, `days_since_last_payment`, `action_payment_plan`
and `promised_to_pay` are the top 4 under every seed; the fifth is a debt-age field. Three complete runs of this notebook
gave the same feature-set file.

## Findings (checked: two full runs give identical results; top-5 overlap 0.8–1.0 across three other seeds)

**Data and setup are sound:** all guards pass (no answer-like columns, synthetic train rows only, no exact duplicates;
`ever_paid` dropped as an exact copy of the never-paid flag). Penalties are chosen inside the grid, not at its edge.

### Feature tiers

| Tier | Pre-call | Post-call (adds) | Evidence |
|---|---|---|---|
| **1 · keep** | `n_payments_90d`, `days_since_last_payment` | `action_payment_plan`, `promised_to_pay` | kept by L1 in 100% of half-samples; top of every method; in the nested top 5 in every fold; the payment fields are **confirmed in real data** (95% CI excludes 0.5) |
| **2 · useful, one per family** | payment-history family: `n_payments_365d`, `paid_share_so_far`, `last_payment_amount`, amounts paid (ρ 0.86–0.98 with each other: one is enough); debt age: `days_since_first_placement`, `days_since_last_placement` | `spoke_with_agent`, `days_since_first_placement` (stable 0.95) | trade places with their twins in L1; LightGBM uses `days_since_last_placement` and `last_payment_amount` (curved effects); payment fields confirmed in real data, debt age not (CI includes 0.5) |
| **3 · little or no value** | account counts, balances, insurance, adjustments, days since discharge | sentiment, QA, transfers, other actions, talk time, inbound | near-zero in every method; none confirmed in real data. Sentiment/QA are low by construction (sampled from the outcome) |

No feature points the opposite way in real data.

### Accuracy: the honest numbers (section 11) correct an earlier claim

| | Pre-call | Post-call |
|---|---|---|
| all candidates | 0.707 (LR) / 0.705 (LightGBM) | 0.797 / 0.795 |
| nested L1 one-SE (≈ 13 / 9 features) | 0.705 / 0.698 | 0.790 / 0.793 |
| nested top 5 | 0.692 / 0.692 | 0.782 / 0.783 |

Chosen and scored on the same rows (section 10), the top 5 looked as good as everything (0.706). **Chosen inside each fold,
5 features lose about 0.015 AUC** against all candidates. So: small sets are close but not free; keeping the whole
payment-history family and debt age is worth it, and adding tier 3 neither helps nor hurts much.

### For the model notebooks

Train each model family on: `stable` (2 / 5), `l1_one_se` (7 / 14), `current` (11 / 32), `top10`, and `all` (21 / 42).
(`elastic_net` keeps every pre-call feature, so it equals `all` there.) Judge them on the synthetic test split and the
real rows; expect differences of about 0.01 between sets on synthetic data.

In [ ]:
out = PHASE2_REPORTS / "experiments" / "feature_sets_master.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"source": MASTER_XLSX.name, "rows": "synthetic train, label_usable (post-call: post_call_row)",
                           "notebook": "13_regularization_feature_importance", "sets": SETS}, indent=2))
print("saved", out); print(json.dumps(SETS, indent=1)[:3000])

**Result:** feature sets saved to `reports/phase2/experiments/feature_sets_master.json` (gitignored; rebuilt by running
this notebook). Summary and recommendations: `feature-selection.md`.